# Score the raw v5 weights with the ablation protocol

Same 3000 random test clips (seed 0), same split, same beam search as `asl-ablate`, so the number is directly comparable to the baseline's 0.399. Also measures how often each test signer's hands are detected, to see whether the hardest signers are a data problem.

In [ ]:
BRANCH = "phase-4-accuracy"
!git clone -q --branch {BRANCH} https://github.com/sohenpatel22/ASL-Fingerspelling-Recognition.git /tmp/repo
%cd /tmp/repo
!pip install -q -e .

In [ ]:
import os, zipfile
from pathlib import Path

def find_dir(name, root="/kaggle/input", max_depth=5):
    for base, dirs, _ in os.walk(root):
        if name in dirs:
            return Path(base, name)
        if base[len(root):].count(os.sep) >= max_depth:
            dirs[:] = []
    raise FileNotFoundError(name)

COMP = next(p for p in (Path("/kaggle/input/competitions/asl-fingerspelling"), Path("/kaggle/input/asl-fingerspelling")) if p.exists())
try:
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train"), find_dir("npy_supp")
except FileNotFoundError:
    for archive in [p for p in Path("/kaggle/input").rglob("*.zip") if "competitions" not in str(p)]:
        with zipfile.ZipFile(archive) as z:
            z.extractall("/tmp/npy")
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train", "/tmp/npy"), find_dir("npy_supp", "/tmp/npy")
DATA = f"data.comp_dir={COMP} data.npy_train={NPY_TRAIN} data.npy_supp={NPY_SUPP}"
print(DATA)

In [ ]:
from asl.checkpoint import fetch_hub_file

V5 = fetch_hub_file("hf:SohenP/asl-fingerspelling-conformer/asl_v5_best.pth")
!asl-eval --checkpoint {V5} --config configs/finetune_v5.yaml --split test --max-samples 3000 --sample-seed 0 --out /kaggle/working/v5_test.json --details-out /kaggle/working/v5_test_details.json {DATA}
assert Path("/kaggle/working/v5_test.json").exists(), "scoring failed: read the output above"
print(open("/kaggle/working/v5_test.json").read())

In [ ]:
import json
import numpy as np
import pandas as pd
from asl.data import participant_split

meta = pd.read_csv(COMP / "train.csv")
_, _, test = participant_split(meta)
rows = {}
for pid, group in test.groupby("participant_id"):
    stats = []
    for _, r in group.sample(min(80, len(group)), random_state=0).iterrows():
        f = NPY_TRAIN / f"{int(r.sequence_id)}.npy"
        if not f.exists():
            continue
        a = np.load(f)
        left, right = (a[:, :42] != 0).any(1), (a[:, 42:] != 0).any(1)
        stats.append((len(a), (left | right).mean(), (left & right).mean(), left.mean(), right.mean(), len(a) / max(len(str(r.phrase)), 1)))
    s = np.array(stats)
    rows[str(pid)] = dict(zip(("frames", "any_hand", "both_hands", "left", "right", "frames_per_char"), map(float, s.mean(0))), n_sequences=len(s))
json.dump(rows, open("/kaggle/working/signer_visibility.json", "w"), indent=1)
print(pd.DataFrame(rows).T.round(2).to_string())